# Benchmark de anomalías
Compara PatchCore y EfficientAD sobre la misma categoría de MVTec AD.


In [ ]:
!pip install -q "anomalib>=2.1.0"


In [ ]:
from pathlib import Path
import pandas as pd
import torch
from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import EfficientAd, Patchcore

DATA_ROOT = "/kaggle/input/mvtec-ad"
CATEGORY = "bottle"
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"

datamodule = MVTecAD(root=DATA_ROOT, category=CATEGORY, train_batch_size=16, eval_batch_size=16, num_workers=2)
print(DEVICE)


In [ ]:
candidates = {
    "patchcore": Patchcore(),
    "efficientad": EfficientAd(),
}
rows = []
for name, model in candidates.items():
    epochs = 1 if name == "patchcore" else 10
    engine = Engine(max_epochs=epochs, accelerator=DEVICE, devices=1)
    engine.fit(datamodule=datamodule, model=model)
    metrics = engine.test(datamodule=datamodule, model=model)
    payload = metrics[0] if metrics else {}
    row = {"model": name}
    for key, value in payload.items():
        try:
            row[str(key)] = float(value)
        except Exception:
            row[str(key)] = str(value)
    rows.append(row)

results = pd.DataFrame(rows)
results


In [ ]:
results.to_csv("/kaggle/working/anomaly_benchmark.csv", index=False)
